# 01 — Generate report labels

CPU, Internet ON, competition attached. Run all cells. Uses Kaggle model quota. See START_HERE.md for saving progress and resuming. Keep the same completed label output attached to every training fold.

## 0 · Setup

In [ ]:
import os, sys, re, glob, json, time, random, threading, subprocess
from dataclasses import dataclass, fields as dc_fields, asdict, is_dataclass
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WORK     = "/kaggle/working"
OUTDIR   = os.path.join(WORK, "labels")
os.makedirs(OUTDIR, exist_ok=True)

assert os.path.isdir(DATA_DIR), (
    f"{DATA_DIR} not found -- use '+ Add Input' to attach the competition dataset. "
    f"Present: {os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'nothing'}"
)
print("competition files:", sorted(os.listdir(DATA_DIR)))
# Optional: attach a previous label run and set its labels/ directory here.
# Copy cached requests before any API calls; never silently overwrite current progress.
RESUME_LABEL_DIR = ""
if RESUME_LABEL_DIR:
    import shutil
    from pathlib import Path
    previous = Path(RESUME_LABEL_DIR)
    assert previous.is_dir(), previous
    for cached in previous.glob('*.jsonl'):
        target = Path(OUTDIR) / cached.name
        if not target.exists(): shutil.copy2(cached, target)

### Authenticating to the model proxy

`kbench.llms` is built from environment variables that `kaggle benchmarks auth`
writes into a `.env` file. From the SDK source:

```python
def is_configured():
    return "MODEL_PROXY_URL" in os.environ and "MODEL_PROXY_API_KEY" in os.environ
```

If those are missing, `import kaggle_benchmarks` still succeeds but `llms` is `{}`
and `llm` is a stand-in that raises as soon as you call it. **This is where your
Kaggle API token is used** — the CLI authenticates with it and exchanges it for
short-lived model-proxy credentials.

Order matters: authenticate *before* importing the SDK, because the registry is
built once, at import.

In [ ]:
# protobuf: BOTH bounds matter. >=5.29.6 clears the gencode mismatch; <6 is
# required by kaggle-benchmarks itself and by grpcio-tools, ydf,
# google-ai-generativelanguage and the google-cloud-* stack. Plain
# "protobuf>=5.29.6" resolves to 7.x and breaks all of them.
!pip install -q "protobuf>=5.29.6,<6" kaggle_benchmarks

In [ ]:
!pip install -q --upgrade kaggle
print()

In [ ]:
# Credential lookup order, straight from kagglesdk/kaggle_env.py:
#   1. in a Kaggle notebook: the file named by $KAGGLE_API_V1_TOKEN  <- usually automatic
#   2. $KAGGLE_API_TOKEN            (the token string itself, or a path to it)
#   3. ~/.kaggle/access_token       (new-style token)
#   4. ~/.kaggle/kaggle.json  or  $KAGGLE_USERNAME + $KAGGLE_KEY   (legacy)
def find_kaggle_creds():
    v1 = os.environ.get("KAGGLE_API_V1_TOKEN")
    if v1 and os.path.exists(v1):
        return f"notebook token ($KAGGLE_API_V1_TOKEN -> {v1})"
    if os.environ.get("KAGGLE_API_TOKEN"):
        return "$KAGGLE_API_TOKEN"
    for f in ("~/.kaggle/access_token", "~/.kaggle/access_token.txt",
              "~/.kaggle/kaggle.json"):
        if os.path.exists(os.path.expanduser(f)):
            return f
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return "$KAGGLE_USERNAME + $KAGGLE_KEY"
    return None


print("kaggle env vars present:", sorted(k for k in os.environ if k.startswith("KAGGLE_")))
found = find_kaggle_creds()
print("credentials:", found or "NONE FOUND")

# Only needed if nothing was found. Add ONE of these under Add-ons > Secrets:
#   KAGGLE_API_TOKEN  = a token from Settings > API Tokens ("Create New Token")
#   KAGGLE_USERNAME + KAGGLE_KEY = the two fields inside a legacy kaggle.json
if not found:
    try:
        from kaggle_secrets import UserSecretsClient
        _sec = UserSecretsClient()
        for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                v = _sec.get_secret(name)
                if v:
                    os.environ[name] = v.strip()
                    print(f"  loaded {name} from Secrets")
            except Exception as e:
                print(f"  {name}: not set ({type(e).__name__})")
    except Exception as e:
        print("  Kaggle Secrets unavailable:", type(e).__name__)
    print("credentials now:", find_kaggle_creds() or "STILL NONE")

In [ ]:
import tempfile
_AUTH_DIR = tempfile.mkdtemp(prefix="knee-auth-")
ENV_FILE = os.path.join(_AUTH_DIR, ".env")
os.chdir(WORK)


def _kaggle(*args):
    # console script first, then the module (a fresh pip install may not be on PATH)
    for exe in (["kaggle"], [sys.executable, "-m", "kaggle.cli"]):
        try:
            return subprocess.run(exe + list(args), capture_output=True, text=True)
        except FileNotFoundError:
            continue
    return subprocess.CompletedProcess(
        [], 127, "", "kaggle CLI not found -- restart the session and re-run.")


# 'auth' writes ONLY MODEL_PROXY_URL / _API_KEY / _EXPIRY_TIME.
# 'init' writes those PLUS LLM_DEFAULT, LLM_DEFAULT_EVAL and LLMS_AVAILABLE.
# kbench.llms is built from LLMS_AVAILABLE, and importing kaggle_benchmarks
# raises KeyError('LLM_DEFAULT') if that one is missing -- so 'init' is required.
proc = _kaggle("benchmarks", "init", "-y",
               "--env-file", ENV_FILE,
               "--example-file", os.path.join(_AUTH_DIR, "example.py"))
print(proc.stdout or "", proc.stderr or "")

import dotenv
dotenv.load_dotenv(ENV_FILE, override=True)

# Safety net: if init didn't populate the model list, use the CLI's own defaults.
if not os.environ.get("LLM_DEFAULT"):
    print("init did not set the model list; applying the CLI's built-in defaults\n")
    os.environ["LLM_DEFAULT"] = "google/gemini-3-flash-preview"
    os.environ["LLM_DEFAULT_EVAL"] = "google/gemini-3-flash-preview"
    os.environ["LLMS_AVAILABLE"] = (
        "anthropic/claude-sonnet-5@default,deepseek-ai/deepseek-r1-0528,"
        "google/gemini-3-flash-preview,google/gemini-3.1-flash-lite-preview,"
        "ibm/granite-4.0-h-small,openai/gpt-5.4-nano-2026-03-17,"
        "openai/gpt-oss-120b,qwen/qwen3-next-80b-a3b-instruct")

for k in ("MODEL_PROXY_URL", "MODEL_PROXY_EXPIRY_TIME", "LLM_DEFAULT", "LLMS_AVAILABLE"):
    v = os.environ.get(k)
    print(f"{k:<24}", (v[:150] if v else "NOT SET"))

In [ ]:
import kaggle_benchmarks as kbench

print("kaggle_benchmarks", getattr(kbench, "__version__", "(n/a)"))
print("proxy configured  :", "MODEL_PROXY_URL" in os.environ)

AVAILABLE = dict(kbench.llms)
print()
print(f"models available: {len(AVAILABLE)}")
for m in sorted(AVAILABLE):
    print("  ", m)

if not AVAILABLE:
    print()
    print("!! llms is empty -> the proxy env vars are not set in THIS process.")
    print("   If the auth cell just succeeded, restart the session and run again:")
    print("   the registry is built once, at import time.")

## 1 · Load everything

The twelve label columns, the reports, the per-series descriptors, and the DICOM
folder for every series. Labels stay `float` with `NaN` for unlabelled studies —
never filled with 0, which would inject thousands of false negatives.

In [ ]:
LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
SERIES_META_COLS = ["Fluid_Sensitive", "Fat_Suppression", "Anatomical_Plane"]
PLANES = ["Sagittal", "Coronal", "Axial"]


def _csv(name):
    return pd.read_csv(os.path.join(DATA_DIR, name))


def add_series_paths(series_df, split, count_slices=False):
    root = os.path.join(DATA_DIR, f"{split}_series")
    df = series_df.copy()
    df["series_dir"] = [os.path.join(root, st, se)
                        for st, se in zip(df["StudyInstanceUID"], df["SeriesInstanceUID"])]
    df["exists"] = [os.path.isdir(p) for p in df["series_dir"]]
    if count_slices:
        df["n_slices"] = [len(glob.glob(os.path.join(p, "*.dcm"))) if e else 0
                          for p, e in zip(df["series_dir"], df["exists"])]
    return df


def study_series_features(series_df):
    # One row per study: series counts by plane and by contrast flag.
    df = series_df.copy()
    df["Anatomical_Plane"] = df["Anatomical_Plane"].astype(str)

    plane = (pd.crosstab(df["StudyInstanceUID"], df["Anatomical_Plane"])
               .reindex(columns=PLANES, fill_value=0).add_prefix("n_"))

    spec = dict(
        n_series=("SeriesInstanceUID", "nunique"),
        n_fluid=("Fluid_Sensitive", lambda s: int(pd.to_numeric(s, errors="coerce").fillna(0).sum())),
        n_fatsat=("Fat_Suppression", lambda s: int(pd.to_numeric(s, errors="coerce").fillna(0).sum())),
    )
    if "n_slices" in df.columns:
        spec["n_slices_total"] = ("n_slices", "sum")

    agg = df.groupby("StudyInstanceUID").agg(**spec)
    return agg.join(plane, how="left").fillna(0).astype(int).reset_index()


def load_all(count_slices=False):
    train  = _csv("train.csv")
    test   = _csv("test.csv")
    sub    = _csv("sample_submission.csv")
    tr_ser = add_series_paths(_csv("train_series.csv"), "train", count_slices)
    te_ser = add_series_paths(_csv("test_series.csv"),  "test",  count_slices)

    for c in LABEL_COLS:
        train[c] = pd.to_numeric(train.get(c), errors="coerce").astype("float32")
    train["has_labels"] = train[LABEL_COLS].notna().all(axis=1)

    for df in (tr_ser, te_ser):
        df["Fluid_Sensitive"] = pd.to_numeric(df["Fluid_Sensitive"], errors="coerce").astype("Int8")
        df["Fat_Suppression"] = pd.to_numeric(df["Fat_Suppression"], errors="coerce").astype("Int8")

    train = train.merge(study_series_features(tr_ser), on="StudyInstanceUID", how="left")
    test  = test.merge(study_series_features(te_ser),  on="StudyInstanceUID", how="left")
    return train, test, tr_ser, te_ser, sub


train, test, train_series, test_series, sample_sub = load_all()

labeled   = train[train["has_labels"]].reset_index(drop=True)
unlabeled = train[~train["has_labels"]].reset_index(drop=True)

print(f"train studies : {len(train):,}")
print(f"  gold labels : {len(labeled):,}")
print(f"  report only : {len(unlabeled):,}")
print(f"test studies  : {len(test):,}")
print(f"train series  : {len(train_series):,}  (dirs on disk: {int(train_series['exists'].sum()):,})")
train.head(3)

In [ ]:
# The entire supervised signal you were given. If these counts are tiny, the
# report extraction below is not an optimisation -- it is the only path.
print("gold positives per finding:\n")
print(pd.DataFrame({
    "positives": labeled[LABEL_COLS].sum().astype(int),
    "rate":      labeled[LABEL_COLS].mean().round(3),
}).to_string())

n_rep = int(train["Report"].notna().sum()) if "Report" in train.columns else 0
print(f"\nreports available: {n_rep:,}")

## 2 · The labelling criteria

These decide what counts as positive, so they *are* your labels. The specific calls
worth arguing with: grade-2 intrasubstance meniscal signal is **not** a tear; an
intact ACL graft is **not** an injury; degenerative subchondral oedema is **not** a
contusion. Edit them here and re-run — everything downstream picks the changes up.

In [ ]:
FINDINGS = [
    ("acl", "ACL",
     "Anterior cruciate ligament injury. POSITIVE for complete or partial tear, "
     "rupture, or discontinuity. NEGATIVE if intact, normal, or continuous. An "
     "intact ACL reconstruction/graft is NEGATIVE; a torn or failed graft is POSITIVE."),
    ("mcl", "MCL",
     "Medial collateral ligament injury. POSITIVE for a sprain of any grade, partial "
     "or complete tear, or periligamentous oedema attributed to MCL injury. "
     "NEGATIVE if intact/normal."),
    ("medial_meniscus", "Medial Meniscus",
     "Medial meniscus TEAR. POSITIVE for any described tear (horizontal, radial, "
     "bucket-handle, complex, root), or grade 3 signal reaching an articular surface. "
     "NEGATIVE for grade 1 or grade 2 intrasubstance signal, or mucoid/myxoid "
     "degeneration without a tear. Post-meniscectomy with no residual or recurrent "
     "tear is NEGATIVE."),
    ("lateral_meniscus", "Lateral Meniscus",
     "Lateral meniscus tear, same criteria as the medial meniscus applied laterally. "
     "A discoid meniscus without a tear is NEGATIVE."),
    ("medial_oa", "Medial OA",
     "Osteoarthritis of the MEDIAL tibiofemoral compartment: cartilage loss or "
     "thinning, marginal osteophytes, subchondral sclerosis or cysts in the medial "
     "compartment, or an explicit statement of medial compartment osteoarthritis."),
    ("lateral_oa", "Lateral OA",
     "Osteoarthritis of the LATERAL tibiofemoral compartment, same criteria applied "
     "laterally."),
    ("pf_oa", "PF OA",
     "Patellofemoral osteoarthritis: chondromalacia patellae, patellar or trochlear "
     "cartilage loss or fissuring, patellofemoral osteophytes, or explicit "
     "patellofemoral degenerative change."),
    ("effusion", "Effusion",
     "Joint effusion / excess intra-articular fluid. POSITIVE at any size including "
     "small or mild. NEGATIVE for a trace explicitly called physiologic or normal, "
     "and NEGATIVE if no fluid is mentioned."),
    ("synovitis", "Synovitis",
     "Synovial inflammation. POSITIVE for synovitis, synovial thickening, "
     "hypertrophy or proliferation, synovial enhancement, a distended bursa with "
     "thickened synovial tissue, joint fluid with debris or loose bodies in a "
     "reactive joint, Hoffa's fat pad oedema or inflammation, or an inflamed "
     "plica. A clear, simple effusion described with no synovial or inflammatory "
     "change at all is NEGATIVE, but do not require the word 'synovitis' itself."),
    ("bakers", "Baker's",
     "Baker's cyst, also called popliteal cyst or gastrocnemio-semimembranosus "
     "bursal cyst. POSITIVE if present, including if ruptured."),
    ("contusion", "Contusion",
     "Bone contusion / bone bruise: traumatic bone marrow oedema. NEGATIVE for "
     "subchondral marrow oedema attributed to degenerative or osteoarthritic change, "
     "and NEGATIVE for oedema that is only reactive to a listed fracture unless a "
     "separate contusion is described."),
    ("fracture", "Fracture",
     "Fracture of any bone about the knee: avulsion, Segond, stress or insufficiency, "
     "osteochondral, acute or subacute. An old fully healed fracture is NEGATIVE."),
]

KEYS    = [k for k, _, _ in FINDINGS]
KEY2COL = {k: c for k, c, _ in FINDINGS}
assert [c for _, c, _ in FINDINGS] == LABEL_COLS

STATES = ["definite", "probable", "possible", "absent", "unsure"]
HARD = {"definite": 1.0, "probable": 0.75, "possible": 0.35,
        "absent": 0.0, "unsure": np.nan}     # unsure still abstains -> masked
SOFT = HARD
SYSTEM_PROMPT = """You are a musculoskeletal radiologist labelling knee MRI reports for a dataset.

You will be given the full text of one knee MRI report. Reports come from many \
different countries and may be written in any language. Do not translate; read it \
in whatever language it is written in and classify directly.

For each of the twelve findings below, decide whether that finding is present in \
THIS knee, on THIS study.

Rules:
- "absent" is correct both when the report explicitly negates the finding AND when \
the report simply does not mention it. Radiologists report what they see; an \
unmentioned finding is a negative finding.
- Handle negation carefully. "No evidence of a meniscal tear", "the ACL is intact", \
"sin rotura", "kein Riss" are all NEGATIVE.
- Handle hedging: "possible", "cannot be excluded", "suspected", "questionable" \
-> answer "unsure".
- Judge only the findings as defined. Do not infer one finding from another: an \
effusion does not imply synovitis, osteoarthritis does not imply a meniscal tear.
- Describe the current state. Prior surgery or an old healed injury is not a current \
positive finding unless the report says the abnormality is still present.

The twelve findings:
{criteria}

Answer for every finding with exactly one of:
- "definite": the report states the finding is present, unambiguously.
- "probable": strongly implied, or stated with mild hedging.
- "possible": weak or indirect evidence ("possible", "cannot be excluded",
  "suspected"), or a borderline/minimal form of the finding.
- "absent":   negated, OR not mentioned at all.
- "unsure":   the report does not evaluate that structure at all (limited or
  truncated study). Use this rarely."""

SYSTEM = SYSTEM_PROMPT.format(
    criteria="\n".join(f"- {k}: {desc}" for k, _, desc in FINDINGS))

print(SYSTEM[:900], "\n...\n")
print(f"system prompt: {len(SYSTEM):,} chars (~{len(SYSTEM)//4:,} tokens)")

## 3 · The model call

`kbench.llms[...]` bills your AI quota. `schema=` gives structured output, so the
model returns a parsed object instead of JSON you have to repair — the tolerant
parser below is only a backstop.

The proxy reports **exact** spend in nanodollars, which is what the budget cap uses.

In [ ]:
# The catalogue is whatever your proxy token authorises -- do not hardcode.
# Cheap-and-fast first: this is a constrained classification, not a reasoning task.
MODEL_PREFERENCE = [
    "google/gemini-3.1-flash-lite",
    "google/gemini-3-flash",
    "openai/gpt-5.4-nano",
    "qwen/qwen3-next",
    "openai/gpt-oss",
    "ibm/granite",
    "anthropic/claude-sonnet",
    "deepseek-ai/deepseek",
]


def pick_model(available, preference=MODEL_PREFERENCE):
    names = sorted(available)
    for pref in preference:
        for n in names:
            if n.startswith(pref):
                return n
    return names[0] if names else None


KBENCH_MODEL = pick_model(AVAILABLE)
print("using model:", KBENCH_MODEL or "(none -- falling back to kbench.llm)")


@dataclass
class ReportLabels:
    acl: str
    mcl: str
    medial_meniscus: str
    lateral_meniscus: str
    medial_oa: str
    lateral_oa: str
    pf_oa: str
    effusion: str
    synovitis: str
    bakers: str
    contusion: str
    fracture: str

print("schema fields:", [f.name for f in dc_fields(ReportLabels)])

In [ ]:
"""SDK-reported per-attempt costs. Never substitute another model's prices."""
import threading

_usage_lock = threading.Lock()
USAGE = {'in': 0, 'out': 0, 'calls': 0, 'errors': 0, 'nanodollars': 0,
         'attempts': 0, 'unmetered': 0}
HALT_REQUESTS = threading.Event()


def record_chat_usage(chat):
    # Capture usage even if response parsing or the provider call failed.
    try:
        u = chat.usage
        n_in = getattr(u, 'input_tokens', 0) or 0
        n_out = getattr(u, 'output_tokens', 0) or 0
        nd = getattr(u, 'total_cost_nanodollars', None)
        if nd is None:
            a = getattr(u, 'input_tokens_cost_nanodollars', None)
            b = getattr(u, 'output_tokens_cost_nanodollars', None)
            nd = a + b if a is not None and b is not None else None
        # Zero/default usage is not evidence that a remote attempt was free.
        known = nd is not None and nd > 0
    except Exception:
        n_in = n_out = 0
        nd, known = None, False
    with _usage_lock:
        USAGE['attempts'] += 1
        USAGE['in'] += n_in
        USAGE['out'] += n_out
        USAGE['nanodollars'] += nd if known else 0
        USAGE['unmetered'] += int(not known)
    if not known:
        HALT_REQUESTS.set()


def spent_usd():
    return USAGE['nanodollars'] / 1e9


def cost_status():
    amount = f"${spent_usd():.4f} SDK-reported batch cost"
    if USAGE['unmetered']:
        amount += f"; total UNKNOWN ({USAGE['unmetered']} unmetered attempts)"
    return amount + '; not account quota remaining'


def quota_error(exc):
    message = str(exc).lower()
    return any(word in message for word in
               ('quota', 'resource_exhausted', 'resource exhausted', 'insufficient credit',
                'budget exceeded', 'daily limit', 'monthly limit'))

_llm_cache = {}


def parse_response(text):
    """Tolerant parse: strips fences, falls back to the first {...} block."""
    t = re.sub(r"^```(?:json)?|```$", "", str(text).strip(), flags=re.MULTILINE).strip()
    try:
        obj = json.loads(t)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", t, flags=re.DOTALL)
        if not m:
            raise
        obj = json.loads(m.group(0))
    return {k: (str(obj.get(k, "unsure")).strip().lower()
                if str(obj.get(k, "unsure")).strip().lower() in STATES else "unsure")
            for k in KEYS}


def call_model(report, model=None):
    """One report -> dict of 12 states. Bills the Kaggle AI quota."""
    model = model or KBENCH_MODEL
    key = model or "__default__"
    if key not in _llm_cache:
        # kbench.llm is the proxy's own default; used when the registry is empty.
        _llm_cache[key] = kbench.llms[model] if model else kbench.llm
    llm = _llm_cache[key]

    text = f"{SYSTEM}\n\n--- BEGIN REPORT ---\n{report}\n--- END REPORT ---"
    if HALT_REQUESTS.is_set():
        raise RuntimeError('Requests paused after quota exhaustion or missing cost telemetry')
    with kbench.chats.new('label-request') as chat:
        try:
            out = llm.prompt(text, schema=ReportLabels)
        finally:
            record_chat_usage(chat)

    payload = json.dumps(asdict(out)) if (is_dataclass(out) and not isinstance(out, type)) else str(out)
    rec = parse_response(payload)

    with _usage_lock:
        USAGE['calls'] += 1
    return rec


def call_with_retry(report, retries=5, model=None):
    last = None
    for attempt in range(retries):
        try:
            return call_model(report, model)
        except Exception as e:
            last = e
            if quota_error(e):
                HALT_REQUESTS.set()
                raise
            if HALT_REQUESTS.is_set():
                raise
            if any(c in str(e) for c in ("401", "403", "PERMISSION_DENIED", "quota", "Quota")):
                raise
            time.sleep(min(60, 2 ** attempt + random.random() * 2))
    with _usage_lock:
        USAGE["errors"] += 1
    raise RuntimeError(f"failed after {retries} attempts: {last}")

print("ready")

In [ ]:
# NOTE the underscore: a plain `dt` collides with `dt = time.time() - t0` in the
# preflight and pricing cells, which turns this module into a float.
import datetime as _dtmod

MIN_REAUTH_SEC  = 300
ASSUMED_TTL_SEC = 45 * 60
REFRESH_MARGIN  = 300

_auth_lock = threading.Lock()
_last_auth = time.time()
_refresh_warned = False


def _parse_expiry():
    try:
        raw = (os.environ.get("MODEL_PROXY_EXPIRY_TIME") or "").strip()
        if not raw:
            return None
        s = raw[:-1] if raw.endswith("Z") else raw
        if not re.search(r"[+-]\d{2}:?\d{2}$", s):
            s += "+00:00"
        t = _dtmod.datetime.fromisoformat(s)
        return t if t.tzinfo else t.replace(tzinfo=_dtmod.timezone.utc)
    except Exception:
        return None                      # any surprise -> unknown, never raise


def seconds_left():
    t = _parse_expiry()
    if t is None:
        return None
    try:
        return (t - _dtmod.datetime.now(_dtmod.timezone.utc)).total_seconds()
    except Exception:
        return None


def _due(now):
    left = seconds_left()
    if left is None:
        return (now - _last_auth) >= ASSUMED_TTL_SEC
    return left <= REFRESH_MARGIN


def refresh_proxy_auth(force=False):
    global _last_auth
    now = time.time()
    if not force:
        if now - _last_auth < MIN_REAUTH_SEC:
            return False
        if not _due(now):
            return False
    with _auth_lock:
        now = time.time()
        if not force and (now - _last_auth < MIN_REAUTH_SEC or not _due(now)):
            return False
        p = _kaggle("benchmarks", "auth", "-y", "--env-file", ENV_FILE)
        _last_auth = time.time()
        if p.returncode != 0:
            print("re-auth FAILED:", ((p.stdout or "") + (p.stderr or ""))[:300])
            return False
        dotenv.load_dotenv(ENV_FILE, override=True)
        _llm_cache.clear()
        kbench.llms = {n: _load_model(n)
                       for n in os.environ.get("LLMS_AVAILABLE", "").split(",") if n}
        left = seconds_left()
        print(f"  re-authenticated | expires in "
              f"{'unknown' if left is None else f'{left/60:.0f} min'}")
        return True


_inner_call = call_with_retry


def call_with_retry(report, retries=5, model=None):
    # A refresh problem must NEVER cost a report. Warn once, then carry on.
    global _refresh_warned
    try:
        refresh_proxy_auth()
    except Exception as e:
        if not _refresh_warned:
            _refresh_warned = True
            print(f"  [warn] refresh check failed ({type(e).__name__}: {e}); "
                  f"continuing without auto-refresh")
    return _inner_call(report, retries, model)

## 4 · Preflight

One real report, one real call. Confirms the quota works and measures the true
per-report cost before you commit to thousands.

In [ ]:
reports_all = train.loc[train["Report"].notna(), "Report"].astype(str)
assert len(reports_all), "no reports in train.csv"

sample = reports_all.iloc[0]
print(f"sample report ({len(sample)} chars):\n{sample[:600]}\n{'-'*70}")

USAGE.update({"in": 0, "out": 0, "calls": 0, "errors": 0, "nanodollars": 0, "attempts": 0, "unmetered": 0})
t0 = time.time()
rec = call_with_retry(sample)
dt = time.time() - t0

print(f"\nok in {dt:.1f}s | {USAGE['in']} in / {USAGE['out']} out tokens")
for k in KEYS:
    print(f"   {KEY2COL[k]:<18} {rec[k]}")

print(cost_status())
print('Check Kaggle’s quota panel for remaining daily/monthly credit.')
if HALT_REQUESTS.is_set():
    print('New requests paused: no reliable cost telemetry. Cached labels remain usable.')


## 5 · Batch runner

Resumable and budget-capped. Every result is flushed to JSONL as it arrives, so a
notebook timeout or a hit budget costs only the restart — re-running skips
everything already done and never re-pays for it.

In [ ]:
_write_lock = threading.Lock()
STOP = threading.Event()


def load_cache(path):
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    r = json.loads(line)
                    done[r["StudyInstanceUID"]] = r
                except Exception:
                    continue
    return done


def run_batch(df, out_jsonl, workers=4, max_usd=9.0, model=None):
    STOP.clear()
    df = df[df["Report"].notna() & (df["Report"].astype(str).str.strip() != "")]
    done = load_cache(out_jsonl)
    todo = df[~df["StudyInstanceUID"].isin(done)]
    print(f"{len(done):,} cached, {len(todo):,} to do | SDK-reported batch limit ${max_usd:.2f}")
    if HALT_REQUESTS.is_set():
        print('Requests paused after quota exhaustion or missing telemetry; using cached results.')
        return jsonl_to_frame(out_jsonl)
    if not len(todo):
        return jsonl_to_frame(out_jsonl)

    fh = open(out_jsonl, "a")
    t0 = time.time()

    def work(row):
        if STOP.is_set() or HALT_REQUESTS.is_set():
            return
        uid, report = row
        try:
            rec = call_with_retry(str(report), model=model)
        except Exception as e:
            with _usage_lock:
                USAGE['errors'] += 1
            if quota_error(e): HALT_REQUESTS.set()
            if HALT_REQUESTS.is_set(): STOP.set()
            print(f"  !! {uid[:14]}...: {str(e)[:110]}")
            return
        rec["StudyInstanceUID"] = uid
        with _write_lock:
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            n, spent = USAGE["calls"], spent_usd()
            if n % 50 == 0:
                rate = n / max(time.time() - t0, 1e-9)
                eta = (len(todo) - n) / rate / 60 if rate else 0
                print(f"  {n:,} done | {rate:.1f}/s | {cost_status()} | ~{eta:.0f} min left")
            if HALT_REQUESTS.is_set():
                STOP.set()
                print('New requests stopped; progress saved. Check quota / SDK usage telemetry.')
            if max_usd is not None and spent >= max_usd and not STOP.is_set():
                STOP.set()
                print(f"\n!! budget cap ${max_usd:.2f} reached after {n:,} calls -- stopping cleanly.")
                print("   Progress saved. Re-run this cell (tomorrow if the daily quota is spent).")

    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(work, zip(todo["StudyInstanceUID"], todo["Report"])))
    fh.close()

    n_done = len(load_cache(out_jsonl))
    print(f"\n{n_done:,}/{len(df):,} extracted | {cost_status()} | {USAGE['errors']} errors")
    if n_done < len(df):
        print("incomplete -- just re-run this cell, it resumes from the cache.")
    return jsonl_to_frame(out_jsonl)


def jsonl_to_frame(path, soft=False):
    recs = list(load_cache(path).values())
    if not recs:
        return pd.DataFrame(columns=["StudyInstanceUID", *LABEL_COLS])
    mapping = SOFT if soft else HARD
    rows = []
    for r in recs:
        row = {"StudyInstanceUID": r["StudyInstanceUID"]}
        for k in KEYS:
            row[KEY2COL[k]] = mapping.get(r.get(k, "unsure"), np.nan)
        row["n_unsure"] = sum(1 for k in KEYS if r.get(k) == "unsure")
        rows.append(row)
    df = pd.DataFrame(rows)
    df[LABEL_COLS] = df[LABEL_COLS].astype("float32")
    return df

print("ready")

## 6 · Validate against the gold labels

You have the reports for the gold-labelled studies too, so the extractor can be
scored for free. Treat this as a smoke test that catches gross prompt failures —
if the gold set is small, per-finding F1 is noisy and a clean table is not proof
the labels are good.

In [ ]:
val_path = os.path.join(OUTDIR, "validate.jsonl")
USAGE.update({"in": 0, "out": 0, "calls": 0, "errors": 0, "nanodollars": 0, "attempts": 0, "unmetered": 0})
val_pred = run_batch(labeled, val_path, workers=4, max_usd=1.00)

In [ ]:
def validate(gold, pred):
    m = gold.merge(pred, on="StudyInstanceUID", suffixes=("_gold", "_pred"))
    print(f"scored on {len(m)} studies\n")
    rows = []
    for c in LABEL_COLS:
        y_all = m[f"{c}_gold"].to_numpy(dtype=float)
        p_all = m[f"{c}_pred"].to_numpy(dtype=float)
        keep = ~np.isnan(y_all) & ~np.isnan(p_all)
        y, p = y_all[keep], (p_all[keep] >= 0.5).astype(float)
        tp = float(((y == 1) & (p == 1)).sum()); fp = float(((y == 0) & (p == 1)).sum())
        fn = float(((y == 1) & (p == 0)).sum()); tn = float(((y == 0) & (p == 0)).sum())
        prec = tp / (tp + fp) if tp + fp else np.nan
        rec  = tp / (tp + fn) if tp + fn else np.nan
        f1 = 2 * prec * rec / (prec + rec) if prec and rec and (prec + rec) else np.nan
        rows.append({"finding": c, "gold_pos": int(np.nansum(y_all)), "pred_pos": int(p.sum()),
                     "abstain": int(np.isnan(p_all).sum()),
                     "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
                     "precision": prec, "recall": rec, "f1": f1})
    rep = pd.DataFrame(rows)
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print(rep.round(3).to_string(index=False))
    print(f"\nmacro F1: {rep['f1'].mean():.3f}   (committed cells only)")
    print(f"mean abstentions: {pred['n_unsure'].mean():.2f} / 12 per study")
    weak = rep[rep["f1"] < 0.6]["finding"].tolist()
    if weak:
        print("\nlow agreement -> down-weight or re-word the criteria for:")
        print("  " + ", ".join(weak))
    return rep

val_report = validate(labeled, val_pred)
val_report.to_csv(os.path.join(WORK, "validation_report.csv"), index=False)

### Read some yourself

The single highest-value hour in this project. Scan a dozen reports against what the
model said and look for a *systematic* mistake — a finding it always misses, a
negation form it trips on. Fix the criteria in section 2 and re-run.

In [ ]:
raw = load_cache(val_path)
show = labeled[labeled["StudyInstanceUID"].isin(raw)].head(3)

for _, row in show.iterrows():
    uid = row["StudyInstanceUID"]
    r = raw[uid]
    print("=" * 78)
    print(str(row["Report"])[:1100])
    print("-" * 78)
    for k in KEYS:
        g = row[KEY2COL[k]]
        gs = {1.0: "present", 0.0: "absent"}.get(g, "?")
        flag = "   <-- DISAGREES" if gs != r.get(k) and r.get(k) != "unsure" else ""
        print(f"  {KEY2COL[k]:<18} model={r.get(k):<8} gold={gs}{flag}")
    print()

## 7 · Extract every remaining report

`max_usd=9.0` leaves a margin under the $10 daily cap and drives off the exact spend
the proxy reports. If it stops on budget, re-run this cell tomorrow.

In [ ]:
pseudo_path = os.path.join(OUTDIR, "pseudo.jsonl")
USAGE.update({"in": 0, "out": 0, "calls": 0, "errors": 0, "nanodollars": 0, "attempts": 0, "unmetered": 0})
pseudo = run_batch(unlabeled, pseudo_path, workers=4, max_usd=9.0)
print(f"\n{len(pseudo):,} studies pseudo-labelled")

In [ ]:
cells = len(pseudo) * len(LABEL_COLS)
nan   = int(pseudo[LABEL_COLS].isna().to_numpy().sum())
print(f"abstained cells: {nan:,} / {cells:,} ({nan/max(cells,1):.2%}) -- masked in the loss\n")

print(pd.DataFrame({
    "pseudo_rate": pseudo[LABEL_COLS].mean().round(4),
    "gold_rate":   labeled[LABEL_COLS].mean().round(4),
}).to_string())
# print("\nBig gaps between the two columns are worth explaining before you trust them.")

## 8 · Build the training target

Gold and pseudo stacked into one frame. `NaN` means *not observed* in both cases and
is masked out of the loss, so a study the model abstained on for one finding still
trains the other eleven heads at full strength.

In [ ]:
gold_part = labeled[["StudyInstanceUID", *LABEL_COLS]].copy()
gold_part["source"] = "gold"

pseudo_part = pseudo[["StudyInstanceUID", *LABEL_COLS]].copy()
pseudo_part["source"] = "pseudo"

train_targets = pd.concat([gold_part, pseudo_part], ignore_index=True)

# Gold is what the leaderboard is scored against; pseudo is a proxy for it. Tune.
train_targets["weight"] = np.where(train_targets["source"] == "gold", 5.0, 1.0)

y = train_targets[LABEL_COLS].to_numpy(dtype=np.float32)
observed = ~np.isnan(y)
print(f"{y.shape[0]:,} studies x 12 findings | {observed.sum():,} observed cells "
      f"({observed.mean():.1%})")

out_csv = os.path.join(WORK, "train_targets.csv")
train_targets.to_csv(out_csv, index=False)
pseudo.to_csv(os.path.join(WORK, "pseudo_labels.csv"), index=False)
print("wrote", out_csv)
train_targets.head()

In [ ]:
def masked_bce(logits, targets):
    """BCE over (B, 12) targets containing NaN. NaN cells get zero gradient."""
    import torch, torch.nn.functional as F
    mask = ~torch.isnan(targets)
    if not mask.any():
        return logits.sum() * 0.0
    loss = F.binary_cross_entropy_with_logits(
        logits, torch.nan_to_num(targets, nan=0.0), reduction="none")
    return (loss * mask).sum() / mask.sum()


# Proof it works: take rows that actually contain abstentions.
import torch

rows = np.where(np.isnan(y).any(axis=1))[0][:8]
if len(rows) == 0:
    rows = np.arange(min(8, len(y)))
    print("(no abstained cells in this run -- nothing to mask)")

sub = y[rows]
t = torch.tensor(sub)
lg = torch.zeros_like(t, requires_grad=True)
loss = masked_bce(lg, t)
loss.backward()

nan_mask = np.isnan(sub)
g = lg.grad.numpy()
print(f"rows checked       : {len(rows)}")
print(f"masked (NaN) cells : {int(nan_mask.sum())}")
print(f"observed cells     : {int((~nan_mask).sum())}")
print(f"loss               : {loss.item():.4f}")
print(f"gradient == 0 on every masked cell  : {bool((g[nan_mask] == 0).all())}")
print(f"gradient != 0 on the observed cells : {bool((g[~nan_mask] != 0).any())}")

In [ ]:
import shutil

ENSEMBLE = [
    KBENCH_MODEL,                          
    "anthropic/claude-sonnet-5@default",  
    "openai/gpt-5.4-nano-2026-03-17",      
]
ENSEMBLE = [m for m in dict.fromkeys(ENSEMBLE) if m in AVAILABLE or m == KBENCH_MODEL]
print("ensemble members:", *ENSEMBLE, sep="\n  ")


def member_path(model, split):
    slug = re.sub(r"[^a-z0-9]+", "_", model.lower()).strip("_")
    return os.path.join(OUTDIR, f"{split}_{slug}.jsonl")


for split, done in (("pseudo", pseudo_path), ("validate", val_path)):
    dst = member_path(KBENCH_MODEL, split)
    if os.path.exists(done) and not os.path.exists(dst):
        shutil.copy(done, dst)
        print(f"reused {os.path.basename(done)} -> {os.path.basename(dst)}")
RUN_SONNET = False  # retain cached Sonnet labels; opt in explicitly to new paid requests

In [ ]:
for model in sorted(ENSEMBLE, key=lambda m: 'anthropic/' in m):
    if 'anthropic/' in model and not RUN_SONNET:
        print(f'{model}: new requests disabled; existing labels retained')
        continue
    for split, df in (("validate", labeled), ("pseudo", unlabeled)):
        path = member_path(model, split)
        remaining = len(df) - len(load_cache(path))
        if remaining <= 0:
            print(f"{model} / {split}: complete")
            continue
        print(f"\n=== {model} / {split}: {remaining} to do ===")
        USAGE.update({"in": 0, "out": 0, "calls": 0, "errors": 0, "nanodollars": 0, "attempts": 0, "unmetered": 0})
        run_batch(df, path, workers=4, max_usd=4.0, model=model)

In [ ]:
def consensus(split, members, min_members=2):
    """Mean of the members' binary calls. NaN where too few members committed."""
    frames = {}
    for m in members:
        f = jsonl_to_frame(member_path(m, split))
        if len(f):
            frames[m] = f.set_index("StudyInstanceUID")[LABEL_COLS]
    if not frames:
        raise RuntimeError("no member files found")

    ids = sorted(set.union(*[set(f.index) for f in frames.values()]))
    cube = np.full((len(frames), len(ids), len(LABEL_COLS)), np.nan, dtype=np.float32)
    for i, m in enumerate(frames):
        arr = frames[m].reindex(ids).to_numpy(dtype=np.float32)
        cube[i] = np.where(np.isnan(arr), np.nan, (arr >= 0.5).astype(np.float32))

    n_committed = np.sum(~np.isnan(cube), axis=0)
    with np.errstate(invalid="ignore"):
        mean = np.nanmean(cube, axis=0)
    mean = np.where(n_committed >= min_members, mean, np.nan)

    out = pd.DataFrame(mean, index=ids, columns=LABEL_COLS)
    out.index.name = "StudyInstanceUID"
    out = out.reset_index()
    out["n_models"] = n_committed.min(axis=1)
    unanimous = np.isin(mean, [0.0, 1.0]) | np.isnan(mean)
    out["frac_unanimous"] = unanimous.mean(axis=1)
    return out


members_present = [m for m in ENSEMBLE
                   if os.path.exists(member_path(m, "pseudo"))
                   and len(load_cache(member_path(m, "pseudo")))]
ens_pseudo = consensus("pseudo", members_present)
ens_val = consensus("validate", members_present)

arr = ens_pseudo[LABEL_COLS].to_numpy()
vals, counts = np.unique(arr[~np.isnan(arr)], return_counts=True)
print("consensus values -- the ranking signal the graded scale failed to produce:")
for v, c in zip(vals, counts):
    print(f"  {v:.2f}  {c:>7,}  ({c/counts.sum():5.1%})")

In [ ]:
from sklearn.metrics import roc_auc_score

def auc_table(gold_df, preds):
    g = gold_df.set_index("StudyInstanceUID")[LABEL_COLS]
    rows = []
    for name, pdf in preds.items():
        p = pdf.set_index("StudyInstanceUID")[LABEL_COLS].reindex(g.index)
        aucs = {}
        for c in LABEL_COLS:
            y, s = g[c].to_numpy(float), p[c].to_numpy(float)
            k = ~np.isnan(y) & ~np.isnan(s)
            aucs[c] = (roc_auc_score(y[k], s[k])
                       if k.sum() > 1 and len(np.unique(y[k])) == 2 else np.nan)
        aucs["MACRO"] = np.nanmean(list(aucs.values()))
        rows.append(pd.Series(aucs, name=name))
    return pd.DataFrame(rows)


preds = {m.split("/")[-1][:22]: jsonl_to_frame(member_path(m, "validate"))
         for m in members_present}
preds["ENSEMBLE"] = ens_val

tbl = auc_table(labeled, preds)
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(tbl.round(3).to_string())

best = tbl.drop(index="ENSEMBLE")["MACRO"].max()
print(f"\nbest single {best:.3f} | ensemble {tbl.loc['ENSEMBLE','MACRO']:.3f} "
      f"| delta {tbl.loc['ENSEMBLE','MACRO'] - best:+.3f}")
print("58 studies -- treat any delta under ~0.03 as noise.")

In [ ]:
gold_part = labeled[["StudyInstanceUID", *LABEL_COLS]].copy(); gold_part["source"] = "gold"
consensus_targets = ens_pseudo.set_index("StudyInstanceUID")[LABEL_COLS]
base_targets = pseudo.set_index("StudyInstanceUID")[LABEL_COLS]
combined = consensus_targets.combine_first(base_targets)
ens_part = combined.reset_index(); ens_part["source"] = "pseudo"
assert ens_part[LABEL_COLS].notna().any(axis=1).sum() >= 5, "Too few labelled studies; resume label generation"
print("Consensus labels used where available; base labels fill missing consensus cells.")

train_targets = pd.concat([gold_part, ens_part], ignore_index=True)
train_targets["weight"] = 1.0                       # gold held out, not up-weighted
train_targets.to_csv(os.path.join(WORK, "train_targets.csv"), index=False)

y = train_targets[LABEL_COLS].to_numpy(dtype=np.float32)
print(f"{y.shape[0]:,} studies | observed cells {(~np.isnan(y)).mean():.1%}")
print(train_targets[train_targets.source=='pseudo'][LABEL_COLS].mean().round(3).to_string())